# 5-4 파라미터 업데이트 흐름 심화

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
observed = ["forward", "loss", "backward", "zero_grad", "step"]
# 필수 의존성만 edge로 두어 zero_grad가 forward보다 먼저라는 권장 배치를 필수조건으로 오해하지 않습니다.
dependencies = [  ### 학습루프에서 반드시 지켜야하는 선후관계 명시
    ("forward", "loss"),
    ("loss", "backward"),
    ("zero_grad", "backward"),
    ("backward", "step"),
]
pos = {name: observed.index(name) for name in observed}
# 관찰 위치를 edge별로 비교하면 실제 위반인 zero_grad→backward만 정확히 드러납니다.
violations = [f"{a} must precede {b}" for a, b in dependencies if pos[a] > pos[b]]
print("violation:", violations[0])
print("recommended:", "zero_grad -> forward -> loss -> backward -> step")

violation: zero_grad must precede backward
recommended: zero_grad -> forward -> loss -> backward -> step


In [2]:
# 검증 가능 정답 코드
import torch
from torch import nn

def train_step(model, batch, loss_fn, optimizer):
    model.train()
    x, y = batch
    # 이전 step의 값을 비운 뒤 이번 loss의 backward를 수행해야 batch 간 의도치 않은 누적을 막습니다.
    optimizer.zero_grad()
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    return loss.item()

torch.manual_seed(2)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
# 참조가 아니라 독립 clone을 저장해야 optimizer.step 전후 weight 변화를 실제 값으로 비교할 수 있습니다.
before = model.weight.detach().clone()
loss_value = train_step(model, (torch.tensor([[1.], [2.]]), torch.tensor([[2.], [4.]])), nn.MSELoss(), optimizer)
after = model.weight.detach().clone()
print("loss_is_float:", isinstance(loss_value, float))
print("weight_changed:", not torch.equal(before, after))

loss_is_float: True
weight_changed: True


In [3]:
# 검증 가능 정답 코드
runs = {
    "A": {
        "calls": ["zero_grad", "forward", "loss", "backward", "step"],
        "weight_before": 1.0, "weight_after": 0.2,
        "loss_before": 4.0, "loss_after": 0.16,
    },
    "B": {
        "calls": ["forward", "loss", "backward", "zero_grad", "step"],
        "weight_before": 1.0, "weight_after": 1.0,
        "loss_before": 4.0, "loss_after": 4.0,
    },
}
audit = {}
# 호출 의존성과 weight·loss 관찰값을 함께 검사해 순서만 맞거나 숫자만 바뀐 실행을 승인하지 않습니다.
for name, run in runs.items():
    pos = {call: run["calls"].index(call) for call in run["calls"]}
    audit[name] = {
        "zero_before_backward": pos["zero_grad"] < pos["backward"],
        "backward_before_step": pos["backward"] < pos["step"],
        "weight_changed": run["weight_after"] != run["weight_before"],
        "loss_decreased": run["loss_after"] < run["loss_before"],
    }
# 네 검증을 모두 통과한 실행만 승인하고 B는 dict 순서상 첫 실패 규칙을 별도로 보고합니다.
approved = [name for name, checks in audit.items() if all(checks.values())]
first_failure_b = next(key for key, passed in audit["B"].items() if not passed)
### next()는 순회 가능한 데이터 "가장 첫 번째 항목 하나만 ##
### 쏙 꺼내오고 끝내는" 파이썬 내장 함수입니다.###
print("audit:", audit)
print("selected:", approved[0] if len(approved) == 1 else "보류")
print("B_first_failure:", first_failure_b)

audit: {'A': {'zero_before_backward': True, 'backward_before_step': True, 'weight_changed': True, 'loss_decreased': True}, 'B': {'zero_before_backward': False, 'backward_before_step': True, 'weight_changed': False, 'loss_decreased': False}}
selected: A
B_first_failure: zero_before_backward
